# Measure your own roofline

This notebook goes with chapter 4.2 (roofline) and chapter 8.2 (tensor cores and arithmetic intensity).
You will measure the memory bandwidth and the FP32 compute rate of your GPU with two tiny kernels, draw the roofline
with matplotlib, and place three kernels on it by their arithmetic intensity.

Run it in Google Colab with a T4 GPU. In the simulator (NUMBA_ENABLE_CUDASIM=1) the code runs and draws a chart, but
the numbers are not real, so the cell falls back to the published T4 figures there.

In [ ]:
import os
import time
import numpy as np
from numba import cuda
import matplotlib.pyplot as plt

SIM = os.environ.get("NUMBA_ENABLE_CUDASIM") == "1"
N = 4_096 if SIM else 1 << 26

def best_ms(fn, reps=5):
    fn()
    cuda.synchronize()
    best = 1e9
    for _ in range(reps):
        t0 = time.perf_counter()
        fn()
        cuda.synchronize()
        best = min(best, (time.perf_counter() - t0) * 1000)
    return best

Line by line, what this cell does and why:

| Line | Code | What and why |
|---|---|---|
| 1 | <code>import os</code> | The simulator switch. |
| 2 | <code>import time</code> | Wall clock timing. |
| 3 | <code>import numpy as np</code> | NumPy. |
| 4 | <code>from numba import cuda</code> | Numba CUDA. |
| 5 | <code>import matplotlib.pyplot as plt</code> | matplotlib draws the chart. |
| 7 | <code>SIM = os.environ.get(&quot;NUMBA_ENABLE_CUDASIM&quot;) == &quot;1&quot;</code> | True in the simulator. |
| 8 | <code>N = 4_096 if SIM else 1 &lt;&lt; 26</code> | 67 million floats per array on a GPU. |
| 10 | <code>def best_ms(fn, reps=5):</code> | Best of a few runs in milliseconds. |
| 11 | <code>fn()</code> | Warm up and compile. |
| 12 | <code>cuda.synchronize()</code> | Wait for the warm up. |
| 13 | <code>best = 1e9</code> | Start with a very large best. |
| 14 | <code>for _ in range(reps):</code> | Timed runs. |
| 15 | <code>t0 = time.perf_counter()</code> | Start the clock. |
| 16 | <code>fn()</code> | Run. |
| 17 | <code>cuda.synchronize()</code> | Wait for the GPU. |
| 18 | <code>best = min(best, (time.perf_counter() - t0) * 1000)</code> | Keep the fastest. |
| 19 | <code>return best</code> | Return it. |

## 1. Bandwidth: a copy kernel

y[i] = x[i] reads 4 bytes and writes 4 bytes per element and does no maths. Its speed is the memory speed.

In [ ]:
@cuda.jit
def copy_kernel(x, y):
    i = cuda.grid(1)
    if i < x.size:
        y[i] = x[i]

d_x = cuda.to_device(np.ones(N, dtype=np.float32))
d_y = cuda.device_array_like(d_x)
ms = best_ms(lambda: copy_kernel[(N + 255) // 256, 256](d_x, d_y))
bw = 8 * N / (ms / 1000) / 1e9
print(f"measured bandwidth: {bw:.0f} GB/s")

Line by line, what this cell does and why:

| Line | Code | What and why |
|---|---|---|
| 1 | <code>@cuda.jit</code> | Compile as a kernel. |
| 2 | <code>def copy_kernel(x, y):</code> | x in, y out. |
| 3 | <code>i = cuda.grid(1)</code> | Global index. |
| 4 | <code>if i &lt; x.size:</code> | Bounds check. |
| 5 | <code>y[i] = x[i]</code> | One read and one write, nothing else. |
| 7 | <code>d_x = cuda.to_device(np.ones(N, dtype=np.float32))</code> | Input on the GPU. |
| 8 | <code>d_y = cuda.device_array_like(d_x)</code> | Output on the GPU. |
| 9 | <code>ms = best_ms(lambda: copy_kernel[(N + 255) // 256, 256](d_x, d_y))</code> | Time the copy. |
| 10 | <code>bw = 8 * N / (ms / 1000) / 1e9</code> | Bytes moved (8 per element) ÷ seconds ÷ 1e9 = GB/s. |
| 11 | <code>print(f&quot;measured bandwidth: {bw:.0f} GB/s&quot;)</code> | A T4 is rated at 320 GB/s. A good copy kernel reaches about 80 to 90 percent of that. |

## 2. Compute: a kernel that is all FMAs

Each thread runs a long loop of fused multiply adds on values in registers. It reads and writes almost nothing, so its
speed is the compute speed. One FMA counts as 2 FLOPs. Four independent chains let the hardware overlap them.

In [ ]:
ITERS = 8 if SIM else 4096

@cuda.jit
def fma_kernel(out):
    i = cuda.grid(1)
    a = np.float32(i) * np.float32(1e-9)
    b, c, d = a + np.float32(1), a + np.float32(2), a + np.float32(3)
    for _ in range(ITERS):
        a = a * np.float32(0.9999) + np.float32(0.0001)
        b = b * np.float32(0.9999) + np.float32(0.0001)
        c = c * np.float32(0.9999) + np.float32(0.0001)
        d = d * np.float32(0.9999) + np.float32(0.0001)
    out[i] = a + b + c + d

T = 1024 * 256 if not SIM else 64
d_o = cuda.device_array(T, dtype=np.float32)
ms = best_ms(lambda: fma_kernel[T // 256 if not SIM else 1, min(T, 256)](d_o))
flops = T * ITERS * 4 * 2
peak = flops / (ms / 1000) / 1e9
print(f"measured FP32 rate: {peak:.0f} GFLOP/s")

Line by line, what this cell does and why:

| Line | Code | What and why |
|---|---|---|
| 1 | <code>ITERS = 8 if SIM else 4096</code> | Loop length. Long on a GPU, so the loop dominates the time. |
| 3 | <code>@cuda.jit</code> | Compile as a kernel. |
| 4 | <code>def fma_kernel(out):</code> | One output per thread, only so the work is not thrown away. |
| 5 | <code>i = cuda.grid(1)</code> | Global index. |
| 6 | <code>a = np.float32(i) * np.float32(1e-9)</code> | A starting value that differs per thread. np.float32 keeps every value in 32 bit floats. |
| 7 | <code>b, c, d = a + np.float32(1), a + np.float32(2), a + np.float32(3)</code> | Three more independent values. Independent chains keep the pipelines full. |
| 8 | <code>for _ in range(ITERS):</code> | The work loop. |
| 9 | <code>a = a * np.float32(0.9999) + np.float32(0.0001)</code> | One FMA: multiply and add in one instruction, 2 FLOPs. |
| 10 | <code>b = b * np.float32(0.9999) + np.float32(0.0001)</code> | Second chain. |
| 11 | <code>c = c * np.float32(0.9999) + np.float32(0.0001)</code> | Third chain. |
| 12 | <code>d = d * np.float32(0.9999) + np.float32(0.0001)</code> | Fourth chain. |
| 13 | <code>out[i] = a + b + c + d</code> | Use all four values, so none of the work can be removed by the compiler. |
| 15 | <code>T = 1024 * 256 if not SIM else 64</code> | 262,144 threads on a GPU, enough to fill every SM many times. |
| 16 | <code>d_o = cuda.device_array(T, dtype=np.float32)</code> | Output array. |
| 17 | <code>ms = best_ms(lambda: fma_kernel[T // 256 if not SIM else 1, min(T, 256)](d_o))</code> | Time it. 1024 blocks of 256 threads on a GPU. |
| 18 | <code>flops = T * ITERS * 4 * 2</code> | Total FLOPs: threads × iterations × 4 chains × 2. |
| 19 | <code>peak = flops / (ms / 1000) / 1e9</code> | GFLOP/s. |
| 20 | <code>print(f&quot;measured FP32 rate: {peak:.0f} GFLOP/s&quot;)</code> | A T4 is rated at about 8,100 GFLOP/s in FP32 at its boost clock. |

## 3. Draw the roofline

Attainable performance = min(peak, bandwidth × AI). The ridge point peak ÷ bandwidth is the arithmetic intensity
where a kernel stops being memory bound.

In [ ]:
if SIM:
    bw, peak = 320.0, 8100.0
ridge = peak / bw
ai = np.logspace(-2, 3, 200)
roof = np.minimum(peak, bw * ai)

kernels = {"vector add": 1 / 12, "SAXPY": 2 / 12, "tiled matmul, TILE 16": 4.0}
plt.figure(figsize=(7, 4.5))
plt.loglog(ai, roof, lw=2, label="roofline")
for name, k_ai in kernels.items():
    plt.scatter([k_ai], [min(peak, bw * k_ai)], zorder=3)
    plt.annotate(name, (k_ai, min(peak, bw * k_ai)), textcoords="offset points", xytext=(6, -12))
plt.axvline(ridge, ls="--", c="grey")
plt.xlabel("arithmetic intensity (FLOP per byte)")
plt.ylabel("GFLOP/s")
plt.title(f"Roofline: {bw:.0f} GB/s, {peak:.0f} GFLOP/s, ridge at {ridge:.1f} FLOP/byte")
plt.legend()
plt.show()

Line by line, what this cell does and why:

| Line | Code | What and why |
|---|---|---|
| 1 | <code>if SIM:</code> | In the simulator our measurements are not real. |
| 2 | <code>bw, peak = 320.0, 8100.0</code> | So use the published T4 figures instead. |
| 3 | <code>ridge = peak / bw</code> | Ridge point in FLOP per byte. About 25 on a T4. |
| 4 | <code>ai = np.logspace(-2, 3, 200)</code> | 200 intensities from 0.01 to 1000 on a log scale. |
| 5 | <code>roof = np.minimum(peak, bw * ai)</code> | The roof: the lower of the compute line and the memory line. |
| 7 | <code>kernels = {&quot;vector add&quot;: 1 / 12, &quot;SAXPY&quot;: 2 / 12, &quot;tiled matmul, TILE 16&quot;: 4.0}</code> | Three kernels and their AI. Vector add: 1 FLOP per 12 bytes. SAXPY: 2 per 12. Tiled matmul with TILE 16 in FP32: 2 × 16 ÷ 8 = 4. |
| 8 | <code>plt.figure(figsize=(7, 4.5))</code> | A new figure. |
| 9 | <code>plt.loglog(ai, roof, lw=2, label=&quot;roofline&quot;)</code> | Log scale on both axes, as rooflines are always drawn. |
| 10 | <code>for name, k_ai in kernels.items():</code> | Place each kernel. |
| 11 | <code>plt.scatter([k_ai], [min(peak, bw * k_ai)], zorder=3)</code> | Its best possible speed is the roof at its AI. |
| 12 | <code>plt.annotate(name, (k_ai, min(peak, bw * k_ai)), textcoords=&quot;offset points&quot;, xytext=(6, -12))</code> | Label the point. |
| 13 | <code>plt.axvline(ridge, ls=&quot;--&quot;, c=&quot;grey&quot;)</code> | A dashed line at the ridge point. |
| 14 | <code>plt.xlabel(&quot;arithmetic intensity (FLOP per byte)&quot;)</code> | Axis label. |
| 15 | <code>plt.ylabel(&quot;GFLOP/s&quot;)</code> | Axis label. |
| 16 | <code>plt.title(f&quot;Roofline: {bw:.0f} GB/s, {peak:.0f} GFLOP/s, ridge at {ridge:.1f} FLOP/byte&quot;)</code> | The title shows the measured numbers. |
| 17 | <code>plt.legend()</code> | The legend. |
| 18 | <code>plt.show()</code> | Draw it. |

## 4. Try these

1. Work out the AI of the naive matmul in notebook 05 and add it to the chart.
2. Repeat the FMA test with float64. The T4 has far fewer FP64 units. Predict the new ridge point first.
3. Use your bandwidth to predict the time of the copy in notebook 03 with pinned memory. Explain the gap with PCIe.